# FSDP：参数 all-gather 与梯度 reduce-scatter

## 为什么需要分析 FSDP 通信

训练显存不只包含权重，还包含梯度、优化器状态、激活和临时 buffer。以 Qwen3-1.7B 为例，bf16 权重约 3.4 GB；某个具体配置能否单卡运行，还取决于优化器精度、序列长度、micro-batch、activation checkpoint 和设备容量，不能只凭模型名断言 OOM。

FSDP（Fully Sharded Data Parallelism）的解决思路是**分片**：把模型参数切成 N 份，每张卡长期只持有 1/N。但分片有代价——计算前必须把参数"拼回来"（all-gather），计算后必须把梯度"切回去"（reduce-scatter）。这两次通信会占用链路带宽、消耗时间。

如果不理解这两次通信的数据量、耗时和在训练循环中的位置，就无法判断训练瓶颈在哪里、overlap 能隐藏多少、以及增加并行度是否划算。本节固定使用 Qwen3-1.7B 一个 Transformer 层，只模拟通信部分——不执行层前向、loss 或优化器更新——来回答三个递进的问题：

> 一次 FSDP 通信搬运多少数据？实际跑出来多快？这些数字在训练循环里意味着什么？

本节只模拟 Qwen3-1.7B 一个 Transformer 层的通信，不执行层前向、loss 或优化器更新。

![FSDP 参数分片与集合通信](images/fsdp_collectives_zh.svg)

**图 1：** 每个 rank 长期只保留参数分片；all-gather 恢复完整参数，reduce-scatter 再切回梯度分片。


## 1. FSDP 原理：分片、收集、规约、切回

### 分片策略

FSDP 切分的是模型状态的**存放位置**，不会改变这一层的数学定义。设完整参数为 $W$，两卡时把它切成 $W_0$ 和 $W_1$：

- 常态下 rank 0 只保存 $W_0$，rank 1 只保存 $W_1$
- 梯度也按相同方式分片——rank $r$ 只保留自己负责的那部分梯度
- 优化器状态同样分片——Adam 的 m 和 v 也只存本地分片对应的部分

这就是 FSDP 能节省长期模型状态显存的原因：常态下每张卡保存约 $1/N$ 的参数、梯度和优化器状态。计算某个 FSDP unit 时，all-gather 会临时恢复该 unit 的完整参数，因此不能说“任何时刻都只有 $1/N$ 参数”。

### 前向：all-gather 临时恢复完整参数

计算前向需要完整 $W$。FSDP 在每层计算前发起 **all-gather**：各 rank 把自己的参数分片发给其他所有 rank，然后**收集和拼接**成完整参数——不做数值求和。每个 rank 拿到的完整 $W$ 是相同的，随后各 rank 用相同参数处理各自的本地 micro-batch。

### 反向：reduce-scatter 规约并切回分片

反向时，每个 rank 根据自己的输入和反向信号，产生一份与完整参数对应的梯度贡献 $G^{(r)}$。因为各 rank 处理的 micro-batch 数据不同，它们的梯度贡献也不同。

**reduce-scatter** 同时做两件事：先对不同 rank 在相同参数位置上的梯度做 SUM（规约），再把规约结果切成 $N$ 份，每个 rank 只留下自己负责的分片（切分）。操作结束后，rank $r$ 只持有自己负责的梯度分片，并据此更新本地的参数分片和优化器状态。参数更新完全是本地的——不需要再通信。

两个操作的核心区别：**all-gather 只做拼接，不做数值运算**——它把各 rank 的分片收集起来拼成完整参数；**reduce-scatter 先做 SUM 规约再切分**——它不仅要搬运数据，还要对梯度做求和。

### 完整循环

因此一层的状态循环可以概括为：

```text
参数分片 → [all-gather] → 临时完整参数 → 前向计算 → 激活
                                                      ↓
参数分片 ← [reduce-scatter] ← 规约后梯度分片 ← 反向计算
```

如果前向后立即释放完整参数（`reshard_after_forward=True`），那么反向计算该层之前，需要再次 all-gather 恢复完整参数。按一层一个 FSDP unit 的简化账本，28 层一共是 28 次前向 AG + 28 次反向 AG + 28 次 RS。反向 AG 可以通过**预取**（prefetch）尝试与计算重叠；实际调用次数和隐藏比例仍要由当前配置的完整训练 trace 验证。

完整参数只在计算窗口内临时存在，不会长期复制在每张卡上——这就是 FSDP 与普通数据并行（DDP）的本质区别。

原理清楚了。下一步量化：通信到底搬运多少数据？

## 2. 数据归属：每个 rank 手里有什么？

以 Qwen3-1.7B 一个 Transformer 层（无 bias）为例，先逐项列出参数，再区分三类数字——逻辑输入、逻辑输出、对端交换量。

- attention：Q/K/V/O = 2048×2048 + 2048×1024 + 2048×1024 + 2048×2048（1024 是 GQA 的 KV 投影宽度，不是序列长度）
- MLP：gate/up/down = 3×2048×6144
- block 的两个 RMSNorm：2×2048
- Q/K 的 head-dim RMSNorm：2×128

bf16 下完整一层 100.7 MB，两卡 FSDP 各持 50.3 MB。

In [ ]:
B, S, H, I = 2, 4096, 2048, 6144
H_Q, H_KV, D = 16, 8, 128
N, dtype_bytes = 2, 2

layer_params = H * (H_Q * D) + 2 * H * (H_KV * D) + H * (H_Q * D) + 3 * H * I + 2 * H + 2 * D
full_bytes = layer_params * dtype_bytes
shard_bytes = full_bytes // N

def mb(x):
    return x / 1_000_000

print(f'layer 参数量：{layer_params:,}')
print(f'完整参数张量：{mb(full_bytes):.3f} MB')
print(f'本地参数分片：{mb(shard_bytes):.3f} MB')
print(f'\nall-gather 逻辑输入（每 rank）：{mb(shard_bytes):.3f} MB')
print(f'all-gather 逻辑输出（每 rank）：{mb(full_bytes):.3f} MB')
print(f'\nreduce-scatter 逻辑输入（每 rank）：{mb(full_bytes):.3f} MB')
print(f'reduce-scatter 逻辑输出（每 rank）：{mb(shard_bytes):.3f} MB')
print(f'\n两个操作的对端交换量：各 {mb(shard_bytes):.3f} MB/rank')

上面算的是参数量。现在把它翻译成通信账本——对 AG 和 RS 分别记录逻辑输入、逻辑输出、对端交换量：

| 操作 | 逻辑输入（每 rank） | 逻辑输出（每 rank） | 每 rank 对端交换量 |
|------|-------------------|-------------------|-------------------|
| all-gather | 本地分片 50.3 MB | 完整参数 100.7 MB | 发送 50.3 MB 给对端 |
| reduce-scatter | 完整梯度 100.7 MB | 规约后分片 50.3 MB | 发送 50.3 MB 给对端 |

这张表说清了两件事：

1. **AG 和 RS 的对端交换量相同**——都是 50.3 MB，两卡时各自把本地数据的一半发给对方。
2. **但逻辑输入不同**——AG 的输入是分片，RS 的输入是完整梯度。只看"传了多少"而不看"传之前手里有什么"，就没法理解为什么需要两个不同的操作。

后面的分析始终把**逻辑张量大小**与**对端交换量**分开记录——Profiler 里这两类数字出现在不同字段，混在一起是分布式通信分析最常见的错误。

## 3. 在两张 NPU 上发起通信并测量

§2 算清了数据量。现在把通信发起来，看实际耗时。

### API 调用

两个 PyTorch 原语分别对应 §1 的两个方向。集合通信需要多进程，直接运行下面的 Bash 单元。

In [ ]:
%%bash
set -euo pipefail

# 需要调度器已分配并暴露两张 NPU。
mkdir -p results
torchrun --standalone --nproc_per_node=2 scripts/fsdp_collectives.py \
  --output-json results/fsdp_latest.json \
  --profile-dir results/profiles/fsdp_latest

### 结果口径

上一个单元把每次、每个 rank 的原始 wall-time 写入 `results/fsdp_latest.json`。对齐同一 iteration 后取两个 rank 的较大值，得到“慢 rank 路径”；下面的表和串行估算全部从该 JSON 自动生成，不再手抄 median、range 或带宽。

有效带宽定义为“每 rank 单向发送 payload / 慢 rank median”。它是本 demo 的有效吞吐，不等于物理链路带宽。`results/profiles/fsdp_latest/` 是另采的一次 trace，只用于确认 HCCL 调用和等待关系，不与计时样本混算。

如果尚未运行 NPU 单元，下一格只提示待采集，不展示历史数字。


In [ ]:
import json
from pathlib import Path

result_path = Path('results/fsdp_latest.json')
if not result_path.exists():
    print('尚无本机结果：请先运行上面的双 NPU 单元。')
else:
    report = json.loads(result_path.read_text())
    by_name = {row['name']: row for row in report['operations']}
    print('operation | slow median | slow P95 | range | rank spread/std | effective GB/s')
    for row in report['operations']:
        slow = row['slow_rank']
        print(
            f"{row['name']} | {slow['median_ms']:.3f} ms | {slow['p95_ms']:.3f} ms | "
            f"{slow['min_ms']:.3f}-{slow['max_ms']:.3f} ms | "
            f"{row['rank_median_spread_ms']:.3f}/{row['rank_median_pstdev_ms']:.3f} ms | "
            f"{slow['effective_gbps']:.3f}"
        )

    ag_ms = by_name['all_gather']['slow_rank']['median_ms']
    rs_ms = by_name['reduce_scatter']['slow_rank']['median_ms']
    print(f'零 overlap 串行估算（reshard=True）：{28 * (2 * ag_ms + rs_ms):.1f} ms/28 blocks')
    print(f'零 overlap 串行估算（保留完整参数）：{28 * (ag_ms + rs_ms):.1f} ms/28 blocks')


## 4. 如何解释前向 AG、反向 RS 与预取

### 前向 all-gather：每层都要付一次

每个 FSDP unit 的前向计算需要完整参数。FSDP 在计算前发起 all-gather，把各 rank 的本地分片拼成完整 $W$。没有 overlap 时，前向 AG 的零 overlap 串行估算是“unit 数 × 本机慢 rank AG median”。

FSDP 可以在当前 unit 计算时提前发起后续 unit 的参数 all-gather。这就是**通信-计算重叠**的核心机制——只有未被计算覆盖的等待才直接延长关键路径。

### 反向 reduce-scatter：同时规约和切分

每个 unit 的反向计算产出梯度贡献。reduce-scatter 对梯度做 SUM 规约并切回分片，也可能与其他 unit 的反向计算重叠。AG 与 RS 谁更慢以及差多少，应从当前 JSON/trace 判断；不能只凭 RS 多了 SUM 就把全部差异归因于规约，因为 HCCL 算法、同步与 rank skew 也会影响耗时。

### 预取：为什么需要 2 次 AG，以及它怎么被隐藏

如果 `reshard_after_forward=True`，前向计算完成后立即释放完整参数，回到只持有分片的状态。反向需要参数时，必须再次 all-gather。这就是每层 2×AG 的来源：

```text
layer N forward:   AG → compute → free params（只留分片）
layer N backward:  AG → compute grad → RS → keep shard
```

FSDP 的预取机制可以在其他 layer 的反向计算期间提前发起后续参数 AG，因此第二次 AG 存在被反向计算覆盖的机会。覆盖多少取决于 unit 粒度、计算量、通信量和调度，不能从独立 collective demo 推断。

预取的前提是 collective 异步发起、且后续计算不立刻等待它。实际效果取决于 layer 计算量与通信量的比例——layer 计算太短时预取来不及完成，通信就暴露在关键路径上。

### Qwen3-1.7B 的 28 层通信账本

两卡时，一次 AG 或 RS 每 rank 单向发送 50.336 MB。`reshard_after_forward=True` 时，每层 2×AG + 1×RS，因此每层发送 151.008 MB，28 个 Transformer blocks 合计发送 **4.228 GB/rank**，接收量大致相同。这个数字不包含 embedding、末尾 norm 等 root group 通信；梯度累积、activation checkpoint 和实际 reshard 策略还可能改变 AG/RS 次数，最终应以 trace 计数为准。

下面用代码验证 28 个 Transformer block 的每 rank 通信量。


In [ ]:
LAYERS = 28

def gb(x):
    return x / 1_000_000_000

# 两卡时，一次 AG 或 RS 的每 rank 对端发送量都等于本地参数分片。
ag_send_per_rank = shard_bytes
rs_send_per_rank = shard_bytes

# reshard_after_forward=True：每层前向 AG + 反向 AG + RS。
reshard_send = LAYERS * (2 * ag_send_per_rank + rs_send_per_rank)

# forward 后保留完整参数：每层只有一次 AG + 一次 RS。
keep_full_send = LAYERS * (ag_send_per_rank + rs_send_per_rank)

print(f'一次 AG 或 RS：每 rank 发送 {mb(shard_bytes):.3f} MB，接收量相同')
print(f'reshard=True：28 层发送 {gb(reshard_send):.3f} GB/rank')
print(f'  接收：{gb(reshard_send):.3f} GB/rank')
print(f'  发送+接收：{gb(2 * reshard_send):.3f} GB/rank')
print(f'保留完整参数：28 层发送 {gb(keep_full_send):.3f} GB/rank')
print(f'  接收：{gb(keep_full_send):.3f} GB/rank')
print(f'  发送+接收：{gb(2 * keep_full_send):.3f} GB/rank')


### 零 overlap 串行估算不是系统上界，也不是 step time

§3 把 28 层的 2×AG+RS 独立中位数直接相加，假设**没有任何重叠**。它只叫“零 overlap 串行估算”：真实训练可能因 overlap 更短，也可能因资源争用、额外通信和 rank skew 更长，因此它不是系统最坏情况上界。实际训练中：

- 预取让部分 AG 与计算重叠
- 反向 RS 可与更早层的计算重叠
- 但第一个 AG（冷启动）和最后一个 RS（收尾）几乎必然暴露在关键路径上
- 通信与计算争用 NPU 资源、rank skew 也可能把实际时间推高

**通信与计算的实际重叠比例、暴露在关键路径上的时间、以及端到端 step time 的变化，见 07.05 §3–§5 的 overlap 与 profiler 验证方法。**


## 5. 从 Demo 到 TorchTitan：同一行代码，框架里发生了什么

§3 的 demo 是手写 `dist.all_gather_into_tensor()` 和 `dist.reduce_scatter_tensor()`。TorchTitan 不让你手写——它用 `fully_shard` 包装 layer，框架自动在 forward 之前插入 all-gather、backward 之后插入 reduce-scatter。这一节把这个"自动"拆开看。

### fully_shard 做了什么

`fully_shard` 是 PyTorch FSDP2 的核心 API。它接收一个 module，做三件事：

1. **切分参数**：把 module 的完整参数切成 $N$ 个分片（$N$ 是 FSDP group 的 rank 数），每个 rank 只持有其中一份。这与 §2 里"完整参数 100.7 MB、本地分片 50.3 MB"是同一件事——`fully_shard` 就是这个切分的执行者。

2. **插入前向通信**：在 module 的 `forward` 之前，自动发起 all-gather，让每个 rank 临时拿到完整参数。module 计算完成后，如果 `reshard_after_forward=True`，立即释放完整参数，回到只持有分片的状态。

3. **插入反向通信**：在 module 的 backward 之后，自动发起 reduce-scatter，把各 rank 算出的完整梯度规约并切回分片。

所以你在 demo 里手动调用的两行 collective，在 TorchTitan 里就是一行 `fully_shard(layer)`——框架替你决定了通信的时机和粒度，训练代码里不需要显式写出任何 collective 调用。

### TorchTitan 怎么把这串起来

```text
1. ParallelDims(fsdp=2)
   → 告诉框架"两张卡做 FSDP"，生成 fsdp_mesh

2. fully_shard(transformer_layer)
   → 执行上面的参数切分 + 插入前向/反向通信

3. HCCL process group
   → 承接 c10d 的 collective 请求，在链路上完成 all-gather / reduce-scatter
```

关键不是 Qwen3 有专用通信 kernel：`fully_shard` 是 PyTorch FSDP2 的通用 API，适用于任何模型。NPU 适配只负责让 HCCL backend 正确承接 c10d 的 collective 请求。

> **软件栈旁注：** `dist.all_gather_into_tensor` 往下经过四层：FSDP 框架（决定时机）→ c10d `_allgather_base_`（通用表示）→ HCCL 适配层 `HcclAllGather`（NPU 映射）→ HCCL 内核 `hcom_allGather`（链路传输）。reduce-scatter 同理。这不是四次通信——是同一个 collective 穿过四层软件栈时，每一层留下了自己的记录。

## 6. 预取（prefetching）对 FSDP 通信的影响

当 `reshard_after_forward=True` 时，反向计算下一层之前还需要一次参数 all-gather。FSDP 可以在当前层计算时，提前为下一层发起这次通信：

![FSDP 参数预取与计算重叠](images/fsdp_prefetch_overlap_zh.svg)

**图：** 当前层反向计算进行时，下一层参数 AG 提前启动；若 AG 在计算结束前完成，就能减少关键路径上的等待。

prefetching 在当前层计算期间提前发起后续参数 AG。它不会减少通信总量，只可能减少关键路径上的等待；收益必须用相同 workload 的完整训练 trace 和 profiler-off 吞吐验证。独立 collective JSON 只提供 payload、慢 rank latency 与零 overlap 串行估算。


## 练习

1. （判断题）FSDP 长期分片保存参数、梯度和优化器状态，但计算某个 FSDP unit 时会临时 all-gather 完整参数。

2. （单选题）FSDP 反向阶段用什么 collective 规约梯度贡献并留下本地分片？
    A. all-gather
    B. reduce-scatter
    C. all-to-all
    D. broadcast

3. （判断题）把独立 all-gather 与 reduce-scatter 的 median 相加，只能得到零 overlap 串行估算，不能称为系统最坏情况上界。

4. （多选题）可信的 FSDP collective 结果应包含哪些信息？
    A. 每个 rank、每次 iteration 的时间
    B. 慢 rank median、P95 与范围
    C. payload 与有效带宽的可复算关系
    D. 只保存一条最快结果

In [ ]:
!cat ./answer/07.02_answer.txt
